In [ ]:
# ======================================================================
# 0. Instalación de Dependencias (Ejecutar solo la primera vez)
# ===============================================================================
# Instalamos las librerías necesarias: pandas, mysql-connector-python, dotenv, requests, ipywidgets.
# Usamos el parámetro -q (quiet) para que la instalación sea silenciosa y no ensucie el cuaderno con texto de descarga.
# Estas dependencias son cruciales para la conexión a MySQL, manejo de datos y la interfaz interactiva.

!pip install -q pandas mysql-connector-python python-dotenv requests ipywidgets

print("Dependencias instaladas y listas para usar.")

In [ ]:
# ==============================================================================
# 1 Celda de configuración: carga .env, conecta a MySQL.
# ==============================================================================
# Esta celda define la función para establecer la conexión a la base de datos MySQL
# utilizando las credenciales cargadas desde el archivo .env.

import os
import json
import requests
import pandas as pd
import mysql.connector
from mysql.connector import Error
from dotenv import load_dotenv
import ipywidgets as widgets
from IPython.display import display, clear_output

# Cargar variables de entorno desde el archivo .env
load_dotenv()

def conectar_bd():
    """Establece y retorna la conexión activa con la base de datos MySQL."""
    try:
        # Intenta conectar usando las variables de entorno
        conexion = mysql.connector.connect(
            host=os.getenv("DB_HOST"),
            port=int(os.getenv("DB_PORT", 3306)), # Casteo a INT y default 3306 por si falla el .env
            database=os.getenv("DB_NAME"),
            user=os.getenv("DB_USER"),
            password=os.getenv("DB_PASSWORD")
        )
        if conexion.is_connected():
            return conexion 
    except Error as e:
        print(f"Error crítico al conectar con MySQL: {e}")
        return None

# Test de conexión: Se realiza una consulta simple para verificar la conectividad
try:
    conexion_test = conectar_bd()
    if conexion_test and conexion_test.is_connected():
        cursor = conexion_test.cursor()
        # Verificamos si la tabla LIBRO existe contando sus registros
        cursor.execute('SELECT COUNT(*) FROM LIBRO;')
        total_libros = cursor.fetchone()[0]
        cursor.close()
        conexion_test.close()

        # Reportamos el resultado de la prueba
        print(f'Conexión exitosa. Libros en BD: {total_libros}')
    else:
        print('Error de conexión: No se pudo conectar a la base de datos.')
except Exception as e:
    print(f'Error de conexión general: {e}')

In [ ]:
# ==============================================================================
# 2 Celda de inspección del esquema: carga structure de tablas para el prompt.
# ==============================================================================
# Esta celda define el esquema completo de la base de datos relacional
# y las vistas optimizadas que se utilizarán como contexto para el LLM.

# Metadatos del modelo relacional y vistas para transferir al contexto del LLM
ESQUEMA_BD = """
Base de datos: BiblioIA (MySQL). Los estados se guardan como INT (FK), usar IDs numéricos en WHERE.

TABLAS DE DOMINIO:
ESTADO_SOCIO(id_estado_socio, nombre) -- 1=Activo, 2=Suspendido, 3=Baja
ESTADO_EJEMPLAR(id_estado_ejemplar, nombre) -- 1=Disponible, 2=Prestado, 3=Dañado, 4=Baja
ESTADO_PRESTAMO(id_estado_prestamo, nombre) -- 1=Activo, 2=Devuelto, 3=Vencido
ESTADO_RESERVA(id_estado_reserva, nombre) -- 1=Pendiente, 2=Avisado, 3=Completada, 4=Cancelada
NACIONALIDAD(id_nacionalidad, nombre)
GENERO(id_genero, nombre, descripcion)

TABLAS PRINCIPALES:
SOCIO(id_socio, dni, nombre, apellido, email, fecha_alta, id_estado_socio) -- default 1
LIBRO(isbn, titulo, anio_publicacion, stock_total, stock_disponible) -- CHECK: 0 <= stock_disponible <= stock_total
AUTOR(id_autor, nombre, apellido, id_nacionalidad)
EJEMPLAR(id_ejemplar, isbn_libro, nro_ejemplar, id_estado_ejemplar) -- default 1; UNIQUE(isbn_libro, nro_ejemplar)
SANCION(id_sancion, id_socio, tipo, fecha_inicio, fecha_fin, motivo)
PRESTAMO(id_prestamo, id_socio, id_ejemplar, fecha_prestamo, fecha_vencimiento, fecha_devolucion, id_estado_prestamo) -- default 1; fecha_devolucion nullable
RESERVA(id_reserva, id_socio, isbn_libro, fecha_solicitud, id_estado_reserva) -- default 1
AUDITORIA_PRESTAMO(id_auditoria, id_prestamo, accion, fecha_cambio, usuario_bd, detalles)

TABLAS INTERMEDIAS (PK compuesta):
LIBRO_AUTOR(isbn_libro, id_autor)
LIBRO_GENERO(isbn_libro, id_genero)

VISTAS (preferir sobre JOINs manuales):
vista_catalogo_libros(isbn, titulo, anio_publicacion, stock_total, stock_disponible, autores, generos)
vista_prestamos_actuales(id_prestamo, dni, socio, libro, nro_ejemplar, fecha_prestamo, fecha_vencimiento, estado_prestamo, dias_mora) -- solo estados 1 y 3
vista_historial_socios(id_socio, dni, socio, isbn, titulo, fecha_prestamo, fecha_devolucion, estado_prestamo)
vista_ranking_libros(isbn, titulo, autores, total_prestamos, stock_disponible) -- orden: total_prestamos DESC
vista_socios_morosos(id_socio, dni, socio, email, prestamos_vencidos, max_dias_mora) -- solo estado 3
vista_socios_activos(id_socio, dni, socio, email, estado_cuenta) -- solo estado_socio 1
vista_reservas_pendientes(id_reserva, socio, libro_solicitado, fecha_solicitud, estado_reserva) -- solo estados 1 y 2

NOTAS:
- 'autores' y 'generos' en vistas son GROUP_CONCAT; filtrar con LIKE '%texto%', no con =.
- Índices disponibles: LIBRO(titulo), SOCIO(dni).
"""

print("Esquema de contexto listo.")


In [ ]:
# ==============================================================================
# 3 Función text_to_sql(pregunta): llama a la API y retorna SQL.
# ==============================================================================
# Esta función es el puente entre el lenguaje natural (pregunta) y el motor SQL.
# Utiliza un prompt de sistema predefinido y ejemplos (few-shot) para guiar al modelo
# a generar consultas SQL válidas para la base de datos.

import os
import requests

# 1. Definimos los ejemplos aislados (Few-Shot Learning)
FEW_SHOT_EXAMPLES = """Pregunta: ¿Cuáles son los 3 libros más prestados de la biblioteca?
SQL: SELECT titulo, total_prestamos FROM vista_ranking_libros LIMIT 3;

Pregunta: Mostrar los títulos de libros publicados en el año 2005
SQL: SELECT titulo FROM LIBRO WHERE anio_publicacion = 2005;

Pregunta: ¿Qué libros escribió Gabriel García Márquez?
SQL: SELECT l.titulo FROM LIBRO l JOIN LIBRO_AUTOR la ON l.isbn = la.isbn_libro JOIN AUTOR a ON la.id_autor =
a.id_autor WHERE a.nombre = 'Gabriel' AND a.apellido = 'García Márquez';

Pregunta: Mostrame cuántos libros hay por cada género
SQL: SELECT g.nombre AS genero, COUNT(lg.isbn_libro) AS cantidad_libros FROM GENERO g LEFT JOIN LIBRO_GENERO lg ON
g.id_genero = lg.id_genero GROUP BY g.nombre;

Pregunta: Listame los socios que tienen préstamos que vencen hoy
SQL: SELECT s.nombre, s.apellido, p.fecha_vencimiento FROM SOCIO s JOIN PRESTAMO p ON s.id_socio = p.id_socio WHERE
p.estado = 'Activo' AND p.fecha_vencimiento = CURDATE();"""

# 2. Armamos el SYSTEM_PROMPT
SYSTEM_PROMPT = f"""
Eres un experto en SQL para MySQL 8.0. Tu única tarea es convertir preguntas en español
a consultas SQL válidas para la base de datos de una biblioteca universitaria.

{ESQUEMA_BD}

{FEW_SHOT_EXAMPLES}

REGLAS ESTRICTAS Y SINTAXIS
1. Responde ÚNICAMENTE con la consulta SQL. Sin explicaciones, sin texto adicional, sin markdown.
2. No uses ```sql ni ``` en tu respuesta.
3. Solo genera consultas SELECT (nunca INSERT, UPDATE, DELETE, DROP).
4. Si la pregunta es ambigua, genera la consulta más útil posible.
5. Usa las VISTAS cuando simplifiquen la consulta.
6. Termina siempre con punto y coma (;).
7. Si la pregunta no especifica una cantidad y la consulta puede devolver muchas filas, usa LIMIT 50 por defecto.
8. Si la pregunta no puede responderse con el esquema dado, responde exactamente: SELECT 'Consulta no soportada' AS
mensaje;
9. Usar CURDATE() para obtener la fecha actual (no NOW() en comparaciones de fecha).
10. Para concatenar texto: CONCAT(campo1, ' ', campo2) AS [nombre_descriptivo].
11. Para agrupar strings: GROUP_CONCAT(DISTINCT ... SEPARATOR ', ') AS [nombre_descriptivo].
12. Los ENUMs se comparan con strings: estado = 'ACTIVO' o estado = 'Suspendido'.
13. Para calcular días de mora: DATEDIFF(CURDATE(), fecha_vencimiento) AS [nombre_descriptivo].
"""

print('Prompt del sistema configurado.')

# 3. La función que hace la llamada a la IA
def text_to_sql(pregunta):
    """Convierte lenguaje natural a SQL usando Ollama con el SYSTEM_PROMPT definido."""
    # Intentamos traer la URL del .env, si no existe usa la local por defecto
    url_ollama = os.getenv("OLLAMA_URL")
    if not url_ollama:
        print("Error: OLLAMA_URL no está definida en el .env")
        return None

    # Concatenamos el contexto maestro con la pregunta específica del usuario
    prompt_final = f"{SYSTEM_PROMPT}\n\nPregunta del usuario a procesar: {pregunta}\nSQL:"

    # Payload para la llamada a la API de Ollama
    payload = {
        "model": "gemma4:e2b",
        "prompt": prompt_final,
        "stream": False,
        "options": {"temperature": 0.0}
    }

    try:
        # Realizamos la solicitud POST a la API de Ollama
        respuesta = requests.post(url_ollama, json=payload)
        respuesta.raise_for_status()

        resultado_json = respuesta.json()
        sql_limpio = resultado_json["response"].strip()

        # Limpieza final de la respuesta del modelo para asegurar que solo se devuelva SQL
        sql_limpio = sql_limpio.replace("```sql", "").replace("```", "").strip()

        # Aseguramos que la respuesta termine con punto y coma
        if not sql_limpio.endswith(";"):
            sql_limpio += ";"

        return sql_limpio

    except requests.exceptions.RequestException as e:
        print(f"Error en la API de Ollama (Fallo de red o HTTP): {e}")
        return None
    except Exception as e:
        print(f"Error inesperado durante la generación de SQL: {e}")
        return None

In [ ]:
# ==============================================================================
# 4 Función ejecutar_consulta(sql): ejecuta en Mysql y retorna DataFrame.
# ==============================================================================
# Esta función se encarga de la ejecución real de la consulta SQL generada
# contra la base de datos MySQL y la conversión de los resultados a un DataFrame de Pandas.

def ejecutar_consulta(sql):
    """Ejecuta sentencias SQL de lectura y extrae los registros en un DataFrame."""
    # Intentamos establecer la conexión a la base de datos
    conexion = conectar_bd()
    if not conexion:
        return None

    try:
        # Usamos dictionary=True para obtener resultados como diccionarios (columnas como claves)
        cursor = conexion.cursor(dictionary=True)
        cursor.execute(sql)
        registros = cursor.fetchall()

        cursor.close()

        # Convertimos los resultados a un DataFrame de Pandas
        return pd.DataFrame(registros)

    except Error as e:
        print(f"Error de ejecución SQL: {e}")
        return None
    finally:
        # Aseguramos que la conexión se cierre siempre
        if conexion and conexion.is_connected():
            conexion.close()

print("Capa de ejecución relacional vinculada.")


In [ ]:
# ==============================================================================
# 5 Función agente_responder(pregunta): orquesta todo y muestra resultado.
# ==============================================================================
# Esta es la función principal que coordina todo el flujo:
# 1. Traducir la pregunta a SQL.
# 2. Ejecutar el SQL en la base de datos.
# 3. Manejar la respuesta (incluyendo la regla de consulta no soportada).
# 4. Renderizar los resultados al usuario.

def agente_responder(pregunta):
    """Punto de entrada principal que coordina la traducción, ejecución y visualización."""

    # Paso 1: Traducción a código de consulta
    sql_generado = text_to_sql(pregunta)

    if not sql_generado:
        print("No se pudo generar la consulta SQL.")
        return None, None

    # INTERCEPCIÓN: Verificamos si el modelo devolvió la respuesta de "Consulta no soportada"
    if "Consulta no soportada" in sql_generado:
        print("El agente indica que la pregunta no está relacionada con la biblioteca o no puede ser respondida con los datos actuales.")
        return None, None

    # Paso 2: Interpela al motor relacional para ejecutar el SQL
    df_resultados = ejecutar_consulta(sql_generado)

    # Paso 3: Renderizado formal de los resultados
    if df_resultados is not None:
        # Mostramos el SQL generado para transparencia
        print(f"SQL Interpretado:\n{sql_generado}\n")

        if df_resultados.empty:
            print("Consulta ejecutada con éxito: No se encontraron registros que coincidan.")
        else:
            print(f"Registros recuperados ({len(df_resultados)} filas):")
            display(df_resultados)
    else:
        print("Error en el proceso de recuperación de datos.")

    return df_resultados, sql_generado

print("Orquestador del agente activo (con manejo de consultas no soportadas).")

In [ ]:
# ==============================================================================
# 6 Demo interactivo: widget ipywidgets para preguntas libres.
# ==============================================================================
# Esta celda crea la interfaz de usuario interactiva en Jupyter Notebook
# que permite al usuario ingresar preguntas en lenguaje natural y ejecutar el agente.

# Creación de componentes UI nativos de Jupyter para la defensa del TP

# Campo de texto donde el usuario ingresa la pregunta
txt_pregunta = widgets.Text(
    value='',
    placeholder='Escribí acá tu consulta...',
    description='Consulta:',
    layout=widgets.Layout(width='70%')
)

# Botón para iniciar la consulta
btn_consultar = widgets.Button(
    description='Preguntar',
    button_style='primary',
    tooltip='Haz clic para traducir y ejecutar',
    icon='search'
)

# Área de salida donde se mostrarán los resultados
area_salida = widgets.Output()

# Función que se ejecuta al hacer clic en el botón
def procesar_click(b):
    """Función que se ejecuta al presionar el botón, orquestando el agente."""
    with area_salida:
        clear_output() # Limpia la pantalla para cada nueva consulta

        pregunta = txt_pregunta.value.strip()

        if pregunta:
            print(f"Procesando: '{pregunta}'...")
            # Llama a la función principal del agente
            df_resultados, sql_generado = agente_responder(pregunta)
        else:
            print("Por favor, escribe una pregunta antes de hacer clic.")

# Asignar la función al evento del botón
btn_consultar.on_click(procesar_click)

# Desactivamos la actualización continua para que solo dispare cuando se presiona Enter
txt_pregunta.continuous_update = False

def on_enter(change):
    """Envía la consulta al presionar Enter en el campo de texto."""
    procesar_click(None)

txt_pregunta.observe(on_enter, 'value')

# Renderizado del panel de control interactivo
if not globals().get('MODO_TEST', False):
    print("PANEL INTERACTIVO BIBLIOIA")
    display(widgets.HBox([txt_pregunta, btn_consultar]), area_salida)

In [ ]:
# ==============================================================================
# 7 Módulo de recomendaciones con IA Generativa (Texto narrativo)
# ==============================================================================
# Esta función es el módulo avanzado que utiliza la IA generativa (Ollama)
# para crear recomendaciones personalizadas y persuasivas ("chamuyo")
# basadas en el historial de lectura del socio y la disponibilidad de libros.

import os
import requests
import pandas as pd
from IPython.display import display, clear_output, Markdown

def recomendar_para(id_socio):
    """
    Analiza el perfil del socio, extrae su historial de lectura, busca un 'Match literario'
    con otros socios, selecciona libros disponibles y utiliza un LLM para redactar recomendaciones narrativas.
    """
    # 1. Obtener nombre del socio
    sql_socio = f"SELECT nombre, apellido FROM SOCIO WHERE id_socio = {id_socio};"
    df_socio = ejecutar_consulta(sql_socio)

    if df_socio is None or df_socio.empty:
        display(Markdown(f"⚠️ No se encontró ningún socio con el ID {id_socio}."))
        return None

    nombre = df_socio.iloc[0]['nombre']
    apellido = df_socio.iloc[0]['apellido']
    df_libros_comun = None
    df_libros_match = None
    match_nombre = None
    match_apellido = None

    display(Markdown(f"## Recomendaciones para {nombre} {apellido}"))
    display(Markdown("---"))

    # 2. Extraer historial de lectura real
    # Consulta para obtener los títulos y géneros de los libros prestados por el socio
    sql_historial = f"""
        SELECT DISTINCT l.titulo, g.nombre AS genero
        FROM PRESTAMO p
        JOIN EJEMPLAR e ON p.id_ejemplar = e.id_ejemplar
        JOIN LIBRO l ON e.isbn_libro = l.isbn
        JOIN LIBRO_GENERO lg ON l.isbn = lg.isbn_libro
        JOIN GENERO g ON lg.id_genero = g.id_genero
        WHERE p.id_socio = {id_socio};
    """
    df_historial = ejecutar_consulta(sql_historial)

    lista_historial = []
    texto_match = ""

    if df_historial is not None and not df_historial.empty:
        display(Markdown("**📚 Tu historial de lectura:**"))
        display(df_historial)

        # Convertimos el historial a una lista de strings para el prompt
        for index, row in df_historial.iterrows():
            lista_historial.append(f"- {row['titulo']} (Género: {row['genero']})")
        texto_historial = "\n".join(lista_historial)

        # --- Lógica de Búsqueda de Match Literario ---
        # Buscamos al socio que haya leído la mayor cantidad de géneros en común
        sql_match = f"""
            SELECT s.nombre, s.apellido, COUNT(DISTINCT g.id_genero) as coincidencias
            FROM PRESTAMO p
            JOIN EJEMPLAR e ON p.id_ejemplar = e.id_ejemplar
            JOIN LIBRO_GENERO lg ON e.isbn_libro = lg.isbn_libro
            JOIN GENERO g ON lg.id_genero = g.id_genero
            JOIN SOCIO s ON p.id_socio = s.id_socio
            WHERE p.id_socio != {id_socio}
              AND g.id_genero IN (
                  SELECT DISTINCT lg2.id_genero
                  FROM PRESTAMO p2
                  JOIN EJEMPLAR e2 ON p2.id_ejemplar = e2.id_ejemplar
                  JOIN LIBRO_GENERO lg2 ON e2.isbn_libro = lg2.isbn_libro
                  WHERE p2.id_socio = {id_socio}
              )
            GROUP BY s.id_socio, s.nombre, s.apellido
            ORDER BY coincidencias DESC
            LIMIT 1;
        """
        df_match = ejecutar_consulta(sql_match)

        if df_match is not None and not df_match.empty:
            match_nombre = df_match.iloc[0]['nombre']
            match_apellido = df_match.iloc[0]['apellido']
            texto_match = f"¡Buenas noticias! Encontramos un 'Match literario' con el socio {match_nombre} {match_apellido}, quien comparte muchos de tus géneros favoritos."

            sql_id_match = f"""
                SELECT s.id_socio
                FROM SOCIO s
                WHERE s.nombre = '{match_nombre}' AND s.apellido = '{match_apellido}'
                LIMIT 1;
            """
            df_id_match = ejecutar_consulta(sql_id_match)

            if df_id_match is not None and not df_id_match.empty:
                id_match = df_id_match.iloc[0]['id_socio']

                # --- Libros que conectaron el match (ambos leyeron) ---
                sql_libros_en_comun = f"""
                    SELECT DISTINCT l.titulo, GROUP_CONCAT(DISTINCT g.nombre SEPARATOR ', ') AS generos
                    FROM PRESTAMO p
                    JOIN EJEMPLAR e ON p.id_ejemplar = e.id_ejemplar
                    JOIN LIBRO l ON e.isbn_libro = l.isbn
                    JOIN LIBRO_GENERO lg ON l.isbn = lg.isbn_libro
                    JOIN GENERO g ON lg.id_genero = g.id_genero
                    WHERE p.id_socio = {id_socio}
                    AND l.isbn IN (
                        SELECT e2.isbn_libro
                        FROM PRESTAMO p2
                        JOIN EJEMPLAR e2 ON p2.id_ejemplar = e2.id_ejemplar
                        WHERE p2.id_socio = {id_match}
                    )
                    GROUP BY l.isbn, l.titulo;
                """
                df_libros_comun = ejecutar_consulta(sql_libros_en_comun)

                # --- Libros que leyó el match y el socio actual no ---
                sql_libros_match = f"""
                    SELECT DISTINCT l.titulo, GROUP_CONCAT(DISTINCT g.nombre SEPARATOR ', ') AS generos
                    FROM PRESTAMO p
                    JOIN EJEMPLAR e ON p.id_ejemplar = e.id_ejemplar
                    JOIN LIBRO l ON e.isbn_libro = l.isbn
                    JOIN LIBRO_GENERO lg ON l.isbn = lg.isbn_libro
                    JOIN GENERO g ON lg.id_genero = g.id_genero
                    WHERE p.id_socio = {id_match}
                    AND l.isbn NOT IN (
                        SELECT e2.isbn_libro
                        FROM PRESTAMO p2
                        JOIN EJEMPLAR e2 ON p2.id_ejemplar = e2.id_ejemplar
                        WHERE p2.id_socio = {id_socio}
                    )
                    AND l.stock_disponible > 0
                    GROUP BY l.isbn, l.titulo
                    LIMIT 5;
                """
                df_libros_match = ejecutar_consulta(sql_libros_match)

                # Armar texto para el prompt de IA con los libros del match
                if df_libros_match is not None and not df_libros_match.empty:
                    libros_match_lista = [f"- {row['titulo']} (Géneros: {row['generos']})" for _, row in df_libros_match.iterrows()]
                    texto_match += f"\n\nLibros que leyó tu match que podrían interesarte:\n" + "\n".join(libros_match_lista)
        else:
            texto_match = "Por el momento no encontramos otro socio con gustos lo suficientemente similares para hacer un Match."

    else:
        # Caso donde el socio es nuevo o no tiene historial
        display(Markdown("*No registras historial de lectura previo.*"))
        texto_historial = "El usuario es nuevo, no tiene historial de lectura previo."
        texto_match = "Al no contar con un historial de lectura, no hemos podido encontrarte un 'Match' con un socio con gustos similares."

    # 3. Extraer candidatos a recomendar (libros disponibles)
    # Buscamos libros que el socio no haya prestado y que tengan stock disponible
    sql_candidatos = f"""
        SELECT l.titulo
        FROM LIBRO l
        WHERE l.stock_disponible > 0
          AND l.isbn NOT IN (
              SELECT e.isbn_libro
              FROM PRESTAMO p
              JOIN EJEMPLAR e ON p.id_ejemplar = e.id_ejemplar
              WHERE p.id_socio = {id_socio}
          )
        ORDER BY RAND() LIMIT 3;
    """
    df_candidatos = ejecutar_consulta(sql_candidatos)

    if df_candidatos is None or df_candidatos.empty:
        display(Markdown("⚠️ No hay libros disponibles para recomendar en este momento."))
        return None

    lista_candidatos = []
    for index, row in df_candidatos.iterrows():
        lista_candidatos.append(f"- {row['titulo']}")
    texto_candidatos = "\n".join(lista_candidatos)

    # 4. Magia de IA: Prompting para la redacción del "Chamuyo"
    display(Markdown("*🤖 El agente IA está analizando tu perfil y redactando las recomendaciones...*"))

    prompt_recomendacion = f"""
    Eres un bibliotecario experto, carismático y amable.
    Tu tarea es recomendarle libros al usuario llamado {nombre}.

    Historial de lectura del usuario:
    {texto_historial}

    Libros que DEBES recomendarle (tenemos stock de estos):
    {texto_candidatos}

    Estado del Match Literario:
    {texto_match}

    Instrucciones estrictas:
    1. Inicia con un saludo entusiasta y personalizado.
    2. Menciona brevemente sus gustos basándote en su historial (si tiene).
    3. Si no tiene historial, suelta un mensaje amigable diciendo que al ser nuevo no puedes recomendar libros basándote en su pasado, y no le ofreces recomendaciones.
    4. Si tiene historial, presenta los libros recomendados. Para cada uno, escribe un párrafo breve (2 líneas) inventando una razón interesante de por qué le gustaría, relacionándolo con lo que ya leyó.
    5. Usa un tono coloquial, amigable y convincente.
    6. No uses código ni markdown complejo, solo texto claro.
    7. Marcar las palabras claves de los libros recomendados en negrita para destacarlos.
    8. MUY IMPORTANTE: Antes de despedirte, incluye un párrafo final comunicando exactamente el "Estado del Match Literario" que te pasé arriba, manteniendo tu tono amable de bibliotecario, y poniendo el nombre de la persona encontrada en el match en negrita.
    9. Incluir emojis relacionados a libros, lectura y emociones para hacer el texto más atractivo. sin abusar de los emojis, solo para darle un toque cálido y visual al texto.
    10. Si no tiene historial de lectura, no empieces el texto de match literario de la forma ' Estado del Match Literario:' sino que directamente comunícalo de forma natural en el texto.
    """

    # Llamada a la API de Ollama para generar el texto final
    url_ollama = os.getenv("OLLAMA_URL")
    if not url_ollama:
        display(Markdown("⚠️ **Error:** OLLAMA_URL no está definida en el .env"))
        return None

    payload = {
        "model": "gemma4:e2b",
        "prompt": prompt_recomendacion,
        "stream": False,
        "options": {"temperature": 0.7}
    }

    try:
        respuesta = requests.post(url_ollama, json=payload)
        respuesta.raise_for_status()
        texto_generado = respuesta.json()["response"].strip()

        # 5. Renderizado final del texto de la IA
        display(Markdown(texto_generado))

        # --- Tablas del match (aparecen después del texto de la IA) ---
        if df_libros_comun is not None and not df_libros_comun.empty:
            display(Markdown(f"### 🤝 Libros que conectaron el match con {match_nombre} {match_apellido}"))
            display(df_libros_comun)
        else:
            display(Markdown(f"### 🤝 Match con {match_nombre} {match_apellido}"))
            display(Markdown("*No leyeron los mismos libros, pero comparten géneros en común.*"))
        if df_libros_match is not None and not df_libros_match.empty:
            display(Markdown(f"### 📖 Libros de {match_nombre} {match_apellido} que todavía no leíste"))
            display(df_libros_match)

    except requests.exceptions.RequestException as e:
        display(Markdown(f"⚠️ **Error al generar texto con Ollama:** {e}"))

    return None

# Ejecutamos la función de prueba para el socio con ID 1
txt_socio = widgets.Text(
    value='1',
    placeholder='Ingresá el ID del socio...',
    description='ID Socio:',
    layout=widgets.Layout(width='40%')
)

btn_recomendar = widgets.Button(
    description='Recomendar',
    button_style='success',
    icon='book'
)

area_recomendacion = widgets.Output()

def procesar_recomendacion(b):
    with area_recomendacion:
        clear_output()
        try:
            id_socio = int(txt_socio.value.strip())
            recomendar_para(id_socio)
        except ValueError:
            print("Por favor ingresá un número de ID válido.")

btn_recomendar.on_click(procesar_recomendacion)

if not globals().get('MODO_TEST', False):
    print("PANEL DE RECOMENDACIONES BIBLIOIA")
    display(widgets.HBox([txt_socio, btn_recomendar]), area_recomendacion)

In [ ]:
# ==============================================================================
# 8. Módulo de alertas de devolución inteligente
# ==============================================================================
# Busca préstamos activos próximos a vencer dentro de un umbral configurable,
# muestra una tabla resumen y usa el LLM para redactar recordatorios personalizados.

import os
import requests
import pandas as pd
from IPython.display import display, clear_output, Markdown

# DataFrame compartido entre los dos botones.
df_alertas_guardado = None

def buscar_alertas(dias=3):
    """
    Ejecuta la consulta SQL y muestra la tabla resumen de préstamos
    próximos a vencer. Almacena los resultados en df_alertas_guardado.

    Parámetros:
        dias (int): Días hacia adelante para buscar vencimientos.
    """
    global df_alertas_guardado

    print(f"🔍 Buscando préstamos que vencen en los próximos {dias} día(s)...")
    print("=" * 55)

    # PRESTAMO no tiene columna 'estado': el estado se filtra por id_estado_prestamo.
    # id_estado_prestamo = 1 corresponde a 'Activo'.
    # fecha_devolucion IS NULL es redundante con estado Activo, pero se mantiene
    # como guarda adicional ante inconsistencias de datos.
    sql_alertas = f"""
        SELECT
            s.id_socio,
            s.nombre,
            s.apellido,
            s.email,
            l.titulo,
            p.fecha_vencimiento,
            DATEDIFF(p.fecha_vencimiento, CURDATE()) AS dias_restantes
        FROM PRESTAMO p
        JOIN EJEMPLAR e ON p.id_ejemplar = e.id_ejemplar
        JOIN LIBRO    l ON e.isbn_libro  = l.isbn
        JOIN SOCIO    s ON p.id_socio    = s.id_socio
        WHERE p.id_estado_prestamo = 1
          AND p.fecha_devolucion IS NULL
          AND p.fecha_vencimiento BETWEEN CURDATE() AND DATE_ADD(CURDATE(), INTERVAL {dias} DAY)
        ORDER BY p.fecha_vencimiento ASC;
    """
    df_alertas_guardado = ejecutar_consulta(sql_alertas)

    if df_alertas_guardado is None or df_alertas_guardado.empty:
        print(f"✅ No hay préstamos que venzan en los próximos {dias} día(s).")
        btn_generar_mensajes.layout.display = 'none'
        return

    print(f"⚠️ Se encontraron {len(df_alertas_guardado)} préstamo(s) próximos a vencer:\n")
    display(df_alertas_guardado[['nombre', 'apellido', 'email', 'titulo', 'fecha_vencimiento', 'dias_restantes']])

    print("\n¿Querés generar los mensajes personalizados para estos socios?")
    txt_cantidad.layout.display = 'inline-flex'
    btn_generar_mensajes.layout.display = 'inline-flex'


# Índice que recuerda el último socio procesado para poder continuar después.
ultimo_procesado = 0

def generar_mensajes(desde=0):
    """
    Genera mensajes personalizados para los socios a partir del índice 'desde'.
    Procesa solo la cantidad indicada en txt_cantidad y actualiza ultimo_procesado.

    Parámetros:
        desde (int): Índice desde donde retomar la generación. 0 = desde el principio.
    """
    global df_alertas_guardado, ultimo_procesado

    if df_alertas_guardado is None or df_alertas_guardado.empty:
        print("No hay alertas cargadas. Ejecutá primero la búsqueda.")
        return

    url_ollama = os.getenv("OLLAMA_URL")
    if not url_ollama:
        print("Error: OLLAMA_URL no está definida en el .env")
        return

    socios_agrupados = list(df_alertas_guardado.groupby(['id_socio', 'nombre', 'apellido', 'email']))
    total    = len(socios_agrupados)
    cantidad = int(txt_cantidad.value)
    hasta    = min(desde + cantidad, total)

    if desde >= total:
        print("✅ Ya se generaron mensajes para todos los socios.")
        btn_generar_mensajes.layout.display = 'none'
        btn_continuar.layout.display = 'none'
        return

    print("=" * 55)
    print(f"📬 Generando mensajes {desde + 1} al {hasta} de {total}...")
    print("=" * 55)

    for i in range(desde, hasta):
        (id_socio, nombre, apellido, email), grupo = socios_agrupados[i]

        libros_por_vencer = [
            f"- '{row['titulo']}' (vence el {row['fecha_vencimiento'].strftime('%d/%m/%Y')}, "
            f"en {row['dias_restantes']} día(s))"
            for _, row in grupo.iterrows()
        ]
        texto_libros = "\n".join(libros_por_vencer)

        prompt_alerta = f"""
Eres un bibliotecario amable y profesional de una biblioteca universitaria.
Redactá un mensaje de recordatorio de devolución para el siguiente socio.

Datos del socio:
- Nombre: {nombre} {apellido}
- Email: {email}

Libros que debe devolver próximamente:
{texto_libros}

Instrucciones estrictas:
1. El mensaje debe ser cordial, breve y directo.
2. Saludá al socio por su nombre.
3. Mencioná cada libro con su fecha de vencimiento.
4. Recordale amablemente que la devolución a tiempo permite que otros socios accedan al material.
5. Cerrá con una despedida cálida firmando como "Biblioteca UTN FRCU".
6. No uses markdown complejo, solo texto limpio.
7. Usá algún emoji relacionado a libros o tiempo para darle calidez, sin abusar.
"""

        payload = {
            "model": "gemma4:e2b",  # modelo local configurado en Ollama
            "prompt": prompt_alerta,
            "stream": False,
            "options": {"temperature": 0.6}
        }

        try:
            print(f"⏳ Generando mensaje {i + 1} de {total}: {nombre} {apellido}...")
            respuesta = requests.post(url_ollama, json=payload)
            respuesta.raise_for_status()
            mensaje_generado = respuesta.json()["response"].strip()

            display(Markdown(f"---\n### 👤 {nombre} {apellido}\n📧 **Email:** {email}"))
            display(Markdown(mensaje_generado))

        except requests.exceptions.RequestException as e:
            print(f"Error al generar mensaje para {nombre} {apellido}: {e}")

    ultimo_procesado = hasta

    if ultimo_procesado < total:
        restantes = total - ultimo_procesado
        btn_continuar.description = f'Generar {min(cantidad, restantes)} más (quedan {restantes})'
        btn_continuar.layout.display = 'inline-flex'
        btn_generar_mensajes.layout.display = 'none'
    else:
        print(f"\n✅ Se generaron mensajes para todos los {total} socios.")
        btn_generar_mensajes.layout.display = 'none'
        btn_continuar.layout.display = 'none'


# ==============================================================================
# Widgets
# ==============================================================================

txt_dias = widgets.BoundedIntText(
    value=3, min=1, max=30, step=1,
    description='Días:',
    layout=widgets.Layout(width='20%')
)

txt_cantidad = widgets.BoundedIntText(
    value=3, min=1, max=50, step=1,
    description='Cantidad:',
    layout=widgets.Layout(width='22%', display='none')
)

btn_buscar = widgets.Button(
    description='Buscar alertas',
    button_style='warning',
    icon='bell',
    layout=widgets.Layout(height='32px', align_items='center'),
    style=widgets.ButtonStyle(font_size='13px')
)

btn_generar_mensajes = widgets.Button(
    description=' Generar mensajes',
    button_style='success',
    icon='envelope',
    layout=widgets.Layout(display='none', height='32px', align_items='center'),
    style=widgets.ButtonStyle(font_size='13px')
)

btn_continuar = widgets.Button(
    description='Generar más',
    button_style='info',
    icon='arrow-right',
    layout=widgets.Layout(display='none', height='32px', align_items='center', width='auto', min_width='220px'),
    style=widgets.ButtonStyle(font_size='13px')
)

area_tabla    = widgets.Output()
area_mensajes = widgets.Output()


def procesar_busqueda(b):
    """Limpia todo, resetea el índice y ejecuta la búsqueda."""
    global ultimo_procesado
    ultimo_procesado = 0
    btn_generar_mensajes.layout.display = 'none'
    btn_continuar.layout.display        = 'none'
    txt_cantidad.layout.display         = 'none'
    with area_tabla:
        clear_output()
        buscar_alertas(dias=txt_dias.value)
    with area_mensajes:
        clear_output()

def procesar_mensajes(b):
    """Genera la primera tanda de mensajes desde el principio."""
    with area_mensajes:
        clear_output()
        generar_mensajes(desde=0)

def procesar_continuar(b):
    """Continúa la generación desde donde se dejó, sin limpiar los mensajes anteriores."""
    with area_mensajes:
        generar_mensajes(desde=ultimo_procesado)


btn_buscar.on_click(procesar_busqueda)
btn_generar_mensajes.on_click(procesar_mensajes)
btn_continuar.on_click(procesar_continuar)


if not globals().get('MODO_TEST', False):
    print("PANEL DE ALERTAS DE DEVOLUCIÓN")
    display(widgets.HBox([txt_dias, btn_buscar, txt_cantidad, btn_generar_mensajes, btn_continuar]))
    display(area_tabla)
    display(area_mensajes)